# BTS Digital Twin - Round1 Train 30000

Notebook này chỉ làm 3 việc:

- train `30000` iteration trên `round1 public_set`
- render `test_poses.csv`
- chấm điểm với GT ở `test/images`

Kết quả cuối cùng cần giữ lại là thư mục `gs_model/` để dùng cho notebook resume.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!pip install -q gdown plyfile tqdm lpips scikit-image


## Bước 1 - Clone gaussian-splatting


In [ ]:
%cd /kaggle/working
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn
import os
os.environ['GS_REPO'] = '/kaggle/working/gaussian-splatting'
print('GS_REPO =', os.environ['GS_REPO'])


## Bước 2 - Clone repo pipeline


In [ ]:
REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
GIT_BRANCH = 'coordination/round1-status'
GITHUB_TOKEN = ''

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('Đã lấy GITHUB_TOKEN từ Kaggle Secrets')
except Exception:
    pass

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

!rm -rf /kaggle/working/project
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/project
%cd /kaggle/working/project


## Bước 3 - Tự dò dataset round1 public_set


In [ ]:
SCENE = 'hcm0031'  # hcm0031 | hcm0034 | HCM0181 | HCM0193 | HCM0204
DATASET_DRIVE_URL = 'https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link'
DATASET_ROOT_OVERRIDE = ''

import os
from pathlib import Path
import subprocess

expected = {'hcm0031', 'hcm0034', 'HCM0181', 'HCM0193', 'HCM0204'}
candidates = []
if DATASET_ROOT_OVERRIDE:
    candidates.append(Path(DATASET_ROOT_OVERRIDE))
candidates.append(Path('/kaggle/working/project/Dataset/VAI_NVS_DATA/phase1/public_set'))

for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
    if base.exists():
        for p in base.rglob('public_set'):
            try:
                names = {x.name for x in p.iterdir() if x.is_dir()}
            except Exception:
                continue
            if expected <= names:
                candidates.append(p)

DATASET_ROOT = None
for p in candidates:
    if p.is_dir():
        names = {x.name for x in p.iterdir() if x.is_dir()}
        if expected <= names:
            DATASET_ROOT = str(p)
            break

if DATASET_ROOT is None and DATASET_DRIVE_URL:
    raw_zip = Path('/kaggle/working/dataset_round1.zip')
    raw_dir = Path('/kaggle/working/_dataset_round1_raw')
    print('Không thấy dataset mount sẵn, bắt đầu tải từ Google Drive...')
    subprocess.run(['gdown', '--fuzzy', DATASET_DRIVE_URL, '-O', str(raw_zip)], check=True)
    raw_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(raw_zip), '-d', str(raw_dir)], check=True)
    for p in raw_dir.rglob('public_set'):
        if not p.is_dir():
            continue
        try:
            names = {x.name for x in p.iterdir() if x.is_dir()}
        except Exception:
            continue
        if expected <= names:
            DATASET_ROOT = str(p)
            break

assert DATASET_ROOT, 'Không tìm thấy Dataset/VAI_NVS_DATA/phase1/public_set'
os.environ['DATASET_ROOT'] = DATASET_ROOT
print('DATASET_ROOT =', DATASET_ROOT)
print('SCENE =', SCENE)


## Bước 4 - Train 30000, render, chấm điểm


In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT = Path('/kaggle/working/project')
WORK_DIR = PROJECT / 'pipeline' / 'work' / SCENE
MODEL_DIR = WORK_DIR / 'gs_model'
RENDER_DIR = WORK_DIR / 'round1_test_renders_30000'
WORK_DIR.mkdir(parents=True, exist_ok=True)

os.environ['ITERATIONS'] = '30000'
os.environ['ANTIALIASING'] = '1'
os.environ['EXPOSURE_COMP'] = '1'
os.environ['SAVE_FINAL_CHECKPOINT'] = '1'
os.environ.pop('START_CHECKPOINT', None)

subprocess.run(['bash', str(PROJECT / 'pipeline' / 'scripts' / '03_train_3dgs.sh'), SCENE], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'render_round1_test_poses.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--model_dir', str(MODEL_DIR),
    '--iteration', '30000',
    '--out_dir', str(RENDER_DIR),
], check=True)

subprocess.run([
    'python', str(PROJECT / 'pipeline' / 'scripts' / 'eval_round1_metrics.py'),
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', str(RENDER_DIR),
    '--out_csv', str(WORK_DIR / 'eval_round1_metrics_30000.csv'),
], check=True)


## Bước 5 - File cần giữ lại

Tải nguyên thư mục này về rồi upload lên Google Drive để dùng cho notebook resume:

`/kaggle/working/project/pipeline/work/<SCENE>/gs_model`


In [ ]:
print(f'/kaggle/working/project/pipeline/work/{SCENE}/gs_model')
